# Zonal Statistics

**Zonal statistics** is a spatial analysis method that aggregates raster values within defined vector zones.

In other words, for each zone (such as a grid cell or administrative boundary), statistical summaries are computed from the pixel values that fall within it.

The method takes you from a **continuous raster surface** – a population distribution, say – to **one figure per spatial unit**.

Applying zonal statistics to WorldPop data gives statistics for the cells of a regular grid.

## 0. Importing Libraries and Preparing Data

### 0.1. Importing Libraries

In [ ]:
import rasterio
import rasterstats as rs

import osmnx as ox
import geopandas as gpd
from shapely.geometry import box

# cache OSM responses on disk, so repeating a query does not hit the server again
ox.settings.cache_folder = "../../cache"


### 0.2. Preparing Data

Load the Vienna city boundary from OpenStreetMap

In [ ]:
area_name = "Vienna, Austria"

admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

In the previous section we prepared the raster layer: clipped it to the city boundary and reprojected it to a metric coordinate system. Now we load the resulting file – **`vienna_cropped_population_utm.tif`** – and read two things we will need later: its CRS, and the total population it holds. Reading with `masked=True` excludes the NoData pixels, exactly as in the previous section.

In [ ]:
raster_path = "../../data/austria/vienna_cropped_population_utm.tif"

with rasterio.open(raster_path) as dataset:
    raster_crs = dataset.crs
    raster_total = dataset.read(1, masked=True).sum()

print(f"Raster CRS: {raster_crs}")
print(f"Total population in the raster: {round(float(raster_total))}")

## 1. Preparing Zones for Analysis

We will use a **regular grid (fishnet)** covering the city of Vienna as our analysis zones.

### 1.1. Creating a Regular Grid

We reuse the grid function written in the [third module](../module_3/geoprocessing_4.ipynb): it takes a layer (to define the extent) and a cell size in metres, reprojects the data if it is in a geographic CRS, and returns the grid as a `GeoDataFrame`.

In [ ]:
def create_regular_grid(data, cell_size):

    # CRS check
    if data.crs is None:
        raise ValueError("Input data has no CRS defined.")

    if data.crs.is_geographic:
        data = data.to_crs(data.estimate_utm_crs())

    # Bounding box
    minx, miny, maxx, maxy = data.total_bounds

    grid_cells = []
    cell_ids = []

    x = minx
    cell_id = 0  # cell counter

    while x < maxx:
        y = miny
        while y < maxy:
            grid_cells.append(
                box(x, y, x + cell_size, y + cell_size)
            )
            cell_ids.append(cell_id)
            cell_id += 1
            y += cell_size
        x += cell_size

    # GeoDataFrame
    grid = gpd.GeoDataFrame(
        {
            "cell_id": cell_ids,
            "geometry": grid_cells
        },
        crs=data.crs
    )

    return grid

### 1.2. Building the Grid

Create a grid with a cell size of 1000 metres:

In [ ]:
grid = create_regular_grid(admin_border, cell_size=1000)
print(f"Cells created: {len(grid)}")

### 1.3. Visualising the Grid

In [ ]:
grid.explore(tiles="cartodbpositron")

## 2. Preparing Data for Analysis

Before computing zonal statistics, it's important to make sure the raster and vector data share the **same coordinate reference system**.

### 2.1. Checking Coordinate Reference Systems

In [ ]:
print(f"Grid CRS: {grid.crs}")
print(f"Raster CRS: {raster_crs}")

### 2.2. Aligning Coordinate Reference Systems

If the two differ, the vector layer is reprojected to match the raster. Here both are already in the same UTM zone, so the grid is left as it is:

In [ ]:
if grid.crs != raster_crs:
    grid = grid.to_crs(raster_crs)

print(f"Same CRS: {grid.crs == raster_crs}")

## 3. Computing Zonal Statistics

Now we calculate raster value statistics within each grid cell.

### 3.1. Computing the Sum

The `zonal_stats()` function from the `rasterstats` library takes the zones and the raster and returns one set of statistics per zone:

- the first argument – the vector zones (a `GeoDataFrame` or a path to a file);
- the second argument – the raster (a path to a file, or an array with its transform);
- `stats` – which statistics to compute (`sum`, `mean`, `min`, `max`, `count` and others);
- `geojson_out=True` – return the zones themselves with the statistics added to their properties, instead of a plain list of numbers.

NoData pixels are taken from the raster metadata and excluded automatically, so they do not distort the sums.

In [ ]:
stats_features = rs.zonal_stats(
    grid,
    raster_path,
    stats="sum",
    geojson_out=True
)

### 3.2. Converting the Result

With `geojson_out=True` the function returns a list of GeoJSON features, so we turn it back into a `GeoDataFrame`:

In [ ]:
gdf_stats = gpd.GeoDataFrame.from_features(stats_features, crs=raster_crs)
gdf_stats.head()

We now have a GeoDataFrame with the sum of raster values computed for each grid cell.

The grid was built over the bounding box of the city, so some cells lie completely outside the clipped raster and have no value at all (`None`). Let's count them and keep only the cells that actually contain data:

In [ ]:
print(f"Cells without data: {gdf_stats['sum'].isna().sum()}")

gdf_stats = gdf_stats[gdf_stats["sum"].notna()]

print(f"Cells left: {len(gdf_stats)}")

It is also worth checking that the zonal statistics account for the whole raster: the sum over all cells should match the population total we printed when opening the file.

In [ ]:
print(f"Sum of the zonal statistics: {round(float(gdf_stats['sum'].sum()))}")
print(f"Total population in the raster: {round(float(raster_total))}")

The two figures match, so no population was lost along the way – the grid covers the entire raster, and the reprojection in the previous section preserved the counts.

## 4. Calculating Population Density

Now the population density of each grid cell.

In [ ]:
gdf_stats["density"] = gdf_stats["sum"] / (gdf_stats.geometry.area / 1_000_000)

Here:

- `sum` – the estimated population count in the cell
- `area` – the cell area (in m²)
- dividing by 1,000,000 converts m² to km²

**One caveat about the edges.** A cell on the rim of the city is only partly covered by the raster: its `sum` counts the people inside Vienna, but its area is the full square kilometre. About a fifth of the cells here are in that position, covered on average by less than half, so their density comes out roughly twice too low – and the outer ring of the map reads emptier than the city really is.

There is no clean fix that keeps the cells comparable. Clipping them to the boundary would leave the edge cells smaller than the rest, and a density per cell would stop meaning the same thing everywhere – the same trade-off we met in the [third module](../module_3/geoprocessing_4.ipynb). What matters is knowing the effect is there, and not reading the rim of the map as a finding.

## 5. Visualising the Result

In [ ]:
gdf_stats.explore(column="density", tiles="cartodbpositron")

## Summary

In this section we explored the zonal statistics method and applied it to WorldPop data.

We:

- created a regular grid (fishnet) for the study area;
- prepared the data by ensuring a consistent coordinate reference system;
- computed the sum of raster values within each grid cell;
- checked that the cell sums add up to the population total of the raster;
- calculated population density;
- visualised the result.

As a result, we moved from a continuous raster surface to aggregated metrics by spatial unit – a form that is much more convenient for analysis and interpretation.